<a href="https://colab.research.google.com/github/iankurbhatt/resume-analyser-ai/blob/main/resume_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Resume Analyzer

This project analyzes a resume against a job description and provides:

- Extracted skills
- Match percentage
- Missing skills
- Improvement suggestions

Built using Python, NLP, and Machine Learning.

Section 1: Setup & Libraries


In [56]:
# Install necessary libraries
!pip install streamlit pypdf scikit-learn

# Import core modules
import re
from google.colab import files
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Section 2: Data Acquisition & Preprocessing

In [57]:
# Upload resume
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# PDF extraction
reader = PdfReader(filename)
resume_text = ""
for page in reader.pages:
    resume_text += page.extract_text()

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9,. ]', '', text)
    return text

cleaned_resume = clean_text(resume_text)
print("Resume cleaned successfully.")

Saving Ankur Bhatt Senior Data Analyst Resume.pdf to Ankur Bhatt Senior Data Analyst Resume (2).pdf
Resume cleaned successfully.


Section 3: Skill Engine

In [58]:
# Define your skillset database
skills_db = [
    "python", "java", "sql", "machine learning", "deep learning",
    "pandas", "numpy", "fastapi", "django", "flask",
    "docker", "aws", "git", "github", "nlp", "tensorflow", "pytorch"
]

def extract_skills(text, skills_list):
    return [skill for skill in skills_list if skill in text]

found_skills = extract_skills(cleaned_resume, skills_db)
print("Skills found:", found_skills)

Skills found: ['python', 'sql', 'pandas', 'numpy', 'git']


Section 4: ML Matching Engine

In [59]:
job_description = """
We are looking for a Python developer with experience in SQL, machine learning, pandas, numpy, and fastapi.
Knowledge of docker, aws, and git is preferred.
"""

# TF-IDF Calculation
documents = [cleaned_resume, job_description]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Cosine Similarity
similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
match_score = round(similarity[0][0] * 100, 2)

# Logic for status and missing skills
status = "Strong match" if match_score >= 75 else "Moderate match" if match_score >= 50 else "Weak match"
job_skills = [s for s in skills_db if s in job_description.lower()]
missing_skills = [s for s in job_skills if s not in found_skills]

Section 5: Final Report

In [60]:
def get_recommendations(missing):
    return "Great match!" if not missing else f"Focus on learning: {', '.join(missing)}"

print("\n" + "="*30)
print(" AI RESUME ANALYSIS REPORT")
print("="*30)
print(f"Match Score: {match_score}%")
print(f"Status: {status}")
print(f"Matched Skills: {found_skills}")
print(f"Missing Skills: {missing_skills}")
print(f"Recommendation: {get_recommendations(missing_skills)}")
print("="*30)


 AI RESUME ANALYSIS REPORT
Match Score: 26.25%
Status: Weak match
Matched Skills: ['python', 'sql', 'pandas', 'numpy', 'git']
Missing Skills: ['machine learning', 'fastapi', 'docker', 'aws']
Recommendation: Focus on learning: machine learning, fastapi, docker, aws


Install streamlit

In [88]:
!pip install streamlit pypdf scikit-learn

Create app file

In [89]:
%%writefile app.py

Overwriting app.py


Install required tools

In [90]:
!pip install streamlit pyngrok

Write Streamlit app

In [97]:
%%writefile app.py

import streamlit as st
from pypdf import PdfReader
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

st.title("AI Resume Analyzer (Full System)")

# Upload resume
uploaded_file = st.file_uploader("Upload Resume PDF")

# Job description
job_description = st.text_area("Paste Job Description")

# Skill database (from your notebook)
skills_db = [
    "python","sql","machine learning","pandas","numpy",
    "fastapi","django","flask","docker","aws","git"
]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

if uploaded_file and job_description:

    # -------------------------
    # 1. Extract resume text
    # -------------------------
    reader = PdfReader(uploaded_file)
    resume_text = ""

    for page in reader.pages:
        resume_text += page.extract_text()

    cleaned_resume = clean_text(resume_text)
    cleaned_job = job_description.lower()

    # -------------------------
    # 2. Skill extraction
    # -------------------------
    found_skills = [s for s in skills_db if s in cleaned_resume]
    job_skills = [s for s in skills_db if s in cleaned_job]

    missing_skills = list(set(job_skills) - set(found_skills))

    # -------------------------
    # 3. ML similarity
    # -------------------------
    docs = [cleaned_resume, cleaned_job]

    tfidf = TfidfVectorizer()
    matrix = tfidf.fit_transform(docs)

    score = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]

    # -------------------------
    # 4. OUTPUT (like your notebook)
    # -------------------------
    st.subheader("Match Score")
    st.write(round(score * 100, 2), "%")

    st.subheader("Matched Skills")
    st.write(found_skills)

    st.subheader("Missing Skills")
    st.write(missing_skills)

    if score > 0.7:
        st.success("Strong Match ✅")
    elif score > 0.4:
        st.warning("Medium Match ⚠️")
    else:
        st.error("Weak Match ❌")

Overwriting app.py


Run Streamlit in background

In [98]:
!streamlit run app.py &>/content/logs.txt &

Start ngrok tunnel

In [99]:
from pyngrok import ngrok

# This will connect to the Streamlit app we just started
public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

Your app is live at: NgrokTunnel: "https://squatted-unpinned-tycoon.ngrok-free.dev" -> "http://localhost:8501"


In [80]:
!pkill -f streamlit
from pyngrok import ngrok
ngrok.kill()